# Popularity Baseline Model 
We defined an activity as popular based on the number of unique postive (rating >= 4) user interactions. <br>
This baseline model establishes a simple benchmark for model performance. <br>  

In [2]:
import pandas as pd 

train_df = pd.read_csv("../data/processed/train.csv")
train_df.head()

,user_id,gmap_id,rating,time
0,100000012795753291799,0x89c25d977025692d:0x378d24e13eb8f34d,4,1512264700675
1,100000012795753291799,0x89c258e28c304997:0xfcafe4e7ce35ee8c,4,1518768933898
2,100000012795753291799,0x89c2590f34e21a53:0x749079380908a830,4,1519751339673
3,100000012795753291799,0x89c25999b9c6eff9:0x4b98becb736928d8,4,1569701541136
4,100000012795753291799,0x89c25cfd883fc003:0xb4ba2be70609711b,4,1570596729750


In [4]:
# positive interaction means rating >= 4
positive_interactions = train_df[train_df["rating"] >= 4]

popularity = (
    positive_interactions
    .groupby("gmap_id")["user_id"]
    .nunique()  
    .reset_index(name="popularity_score")
    .sort_values("popularity_score", ascending=False)
)

popularity.head(10)

,gmap_id,popularity_score
29928,0x89c279b142e9dc41:0xf3bfe6b9d563bad,5774
18697,0x89c25b1087680443:0x178bf7c600df7a98,5613
23495,0x89c25e4bc3a2b003:0x1b2445a846ea9a58,5519
13533,0x89c2599f2db306bb:0x2a5c7f9eea6938a0,5490
26692,0x89c2607fc22166fb:0x133d2c68e8326ce1,5369
7474,0x89c258f55fbea537:0x49d5f074061cac4e,5319
26012,0x89c25fe0b147d195:0xf55b95d4f099763b,5073
1394,0x89c24343ab3c4adf:0xfb7370c49a9db106,5061
14073,0x89c259a6bd917da7:0x79997d3983197d7c,4834
28620,0x89c2644bfae67439:0x7e77a9e355f9aa3,4747


In [5]:
popularity.to_csv("../data/processed/popularity.csv", index=False)

"popularity score" is the number of unique users who had a positive experince with each gmapid location. 

In [10]:
def recommend_popularity(user_id, k=10):

    seen = set(
        train_df[
            train_df["user_id"] == user_id
        ]["gmap_id"]
    )

    recommendations = popularity[
        ~popularity["gmap_id"].isin(seen)
    ].head(k).copy()

    recommendations["user_id"] = user_id

    recommendations["rank"] = range(1, len(recommendations) + 1)

    recommendations = recommendations.rename(
        columns={"popularity_score": "score"}
    )

    return recommendations[
        ["user_id", "gmap_id", "score", "rank"]
    ]

In [ ]:
user = train_df["user_id"].iloc[0] 

recommend_popularity(user, k=10)

,user_id,gmap_id,score,rank
29928,100000012795753291799,0x89c279b142e9dc41:0xf3bfe6b9d563bad,5774,1
18697,100000012795753291799,0x89c25b1087680443:0x178bf7c600df7a98,5613,2
23495,100000012795753291799,0x89c25e4bc3a2b003:0x1b2445a846ea9a58,5519,3
13533,100000012795753291799,0x89c2599f2db306bb:0x2a5c7f9eea6938a0,5490,4
26692,100000012795753291799,0x89c2607fc22166fb:0x133d2c68e8326ce1,5369,5
7474,100000012795753291799,0x89c258f55fbea537:0x49d5f074061cac4e,5319,6
26012,100000012795753291799,0x89c25fe0b147d195:0xf55b95d4f099763b,5073,7
1394,100000012795753291799,0x89c24343ab3c4adf:0xfb7370c49a9db106,5061,8
14073,100000012795753291799,0x89c259a6bd917da7:0x79997d3983197d7c,4834,9
28620,100000012795753291799,0x89c2644bfae67439:0x7e77a9e355f9aa3,4747,10


The popularity baseline will only reccomend the user places they havn't intercated with before, so each list will be slightly different for each user. 